In [11]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [12]:
data = fetch_california_housing()

X = pd.DataFrame(data.data, columns=data.feature_names).values
y = data.target


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [14]:
normalizer = tf.keras.layers.Normalization()
normalizer.adapt(X_train)  # learns mean & variance


In [15]:
BATCH = 32

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = (
    train_ds
    .shuffle(buffer_size=10000)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_ds = (
    test_ds
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)


In [16]:
model = tf.keras.Sequential([
    normalizer,
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1)  # regression output
])


In [17]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=[tf.keras.metrics.RootMeanSquaredError()]
)


In [18]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30
)


Epoch 1/30
516/516 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1.3128 - root_mean_squared_error: 1.1229 - val_loss: 0.4416 - val_root_mean_squared_error: 0.6645
Epoch 2/30
516/516 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.4132 - root_mean_squared_error: 0.6427 - val_loss: 0.3873 - val_root_mean_squared_error: 0.6223
Epoch 3/30
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3713 - root_mean_squared_error: 0.6093 - val_loss: 0.3748 - val_root_mean_squared_error: 0.6122
Epoch 4/30
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3514 - root_mean_squared_error: 0.5927 - val_loss: 0.3683 - val_root_mean_squared_error: 0.6068
Epoch 5/30
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3368 - root_mean_squared_error: 0.5803 - val_loss: 0.3453 - val_root_mean_squared_error: 0.5877
Epoch 6/30
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3404 - root_mean_squared_error: 0.5833 - val_loss: 0.3331 - val_root_mean_squared_error: 0.5772
Epoch 7/30
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 

In [19]:
model.evaluate(test_ds)


129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2827 - root_mean_squared_error: 0.5314


[0.2902708947658539, 0.53876793384552]

In [20]:
new_data = np.array([[5.0, 25, 3000, 500, 1200, 450, 34.2, -118.4]])

prediction = model.predict(new_data)
print("Predicted Median House Value:", prediction[0][0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
Predicted Median House Value: 30.249937
